# Component 1 Walkthrough — GIK Parquet → IceChunk Virtual Store

C1 ingests GIK Parquet key-reference files produced by ECMWF's IFS ensemble output
and commits them as virtual Zarr arrays into an IceChunk store with full time-travel support.

**Key classes**: `GIKFlatParquetParser`, `IceChainStore`

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import zarr
from pathlib import Path
import json, tempfile

WORK_DIR = Path(tempfile.mkdtemp(prefix='gik_c1_'))
print('Working directory:', WORK_DIR)

## 1.1  GIK Parquet Schema

The GIK Parquet format stores chunk references in a flat key-value table:

| key | value |
|-----|-------|
| `step_NNN/{var}/sfc/{member}/{chunk_idx}` | `["s3://bucket/file.grib2", offset, length]` |
| `{var}/{level_type}/{level}/.zarray` | `{ zarr metadata JSON }` |

In [ ]:
NLAT, NLON, NSTEPS = 6, 6, 4
LAT  = np.linspace(0.0, 3.0, NLAT, dtype=np.float32)
LON  = np.linspace(35.0, 38.0, NLON, dtype=np.float32)
STEPS = np.arange(0, NSTEPS * 6, 6, dtype=np.int32)

rows = []
for step in STEPS:
    rows.append({
        'key': f'step_{step}/tp/sfc/0/0',
        'value': json.dumps([f's3://ecmwf-forecasts/fake/{step}.grib2', int(step) * 100, 500]),
    })
rows.append({
    'key': 'tp/heightAboveGround/0/.zarray',
    'value': json.dumps({
        'chunks': [1, NLAT, NLON], 'compressor': None, 'dtype': '<f4',
        'fill_value': 'NaN', 'filters': None, 'order': 'C',
        'shape': [NSTEPS, NLAT, NLON], 'zarr_format': 2,
    }),
})

parquet_path = WORK_DIR / 'sample.parquet'
pd.DataFrame(rows).to_parquet(parquet_path)
pd.read_parquet(parquet_path)

## 1.2  Parsing with GIKFlatParquetParser

The parser extracts step-hour integers and maps chunk refs into a VirtualiZarr ManifestStore.

In [ ]:
try:
    import virtualizarr
    import icechunk
    from gik_icechain.conversion.virtualizer import GIKFlatParquetParser, _SFC_STEP_RE

    df = pd.read_parquet(parquet_path)
    df['key'] = df['key'].astype(str)
    extracted = df['key'].str.extract(_SFC_STEP_RE, expand=True)
    sfc_mask = extracted[0].notna()
    step_hours = sorted(extracted.loc[sfc_mask, 0].astype(int).unique().tolist())
    print(f'SFC step hours found: {step_hours}')
    assert step_hours == sorted(int(s) for s in STEPS)
    print('Step hours correctly extracted from Parquet.')
except ImportError as e:
    print(f'Skipping: {e}')

## 1.3  IceChunk Store — Create, Commit, Tag

In [ ]:
try:
    import icechunk
    from gik_icechain.conversion.icechunk_writer import IceChainStore
    from datetime import date

    store_path = str(WORK_DIR / 'icechunk_store')
    store = IceChainStore(store_path)
    store.create_or_open()

    TEST_DATE = date(2024, 10, 15)
    date_str  = TEST_DATE.isoformat()

    session = store._repo.writable_session(store.branch)
    ds = xr.Dataset({'tp': xr.DataArray(
        np.random.default_rng(0).random((3, NSTEPS, NLAT, NLON), dtype=np.float32),
        dims=['member', 'step', 'latitude', 'longitude'],
        coords={'member': [0, 1, 2], 'step': STEPS, 'latitude': LAT, 'longitude': LON},
    )})
    root = zarr.open_group(session.store, zarr_format=3)
    if date_str in root:
        del root[date_str]
    ds.to_zarr(session.store, group=date_str, mode='w')
    commit_hash = session.commit('C1 walkthrough commit', metadata={'forecast_date': date_str})
    store._repo.create_tag(f'{date_str}T00Z', commit_hash)
    print(f'Committed snapshot: {commit_hash[:12]}...')
except ImportError as e:
    print(f'Skipping: {e}')

## 1.4  Time-Travel Checkout

`checkout_as_of(date)` reads the snapshot tagged for that date.

In [ ]:
try:
    historical = store.checkout_as_of(TEST_DATE)
    print(historical)
    assert 'tp' in historical.data_vars
    print('Time-travel checkout: OK')
except NameError:
    print('store not initialised — icechunk not installed')

## 1.5  Store Validation

In [ ]:
try:
    report = store.validate()
    print(report)
    assert report['committed_days'] >= 1
    assert report['gaps_detected'] == 0
    print('Validation passed.')
except NameError:
    print('store not initialised — icechunk not installed')